# P3 — PREDICT / 제출 파일 생성

공식 지표는 `pooled RMSE (m)`입니다. 이 notebook은 `03_model`의 실제 가중치를 로드해 추론하며, 기존 최고점 CSV를 입력으로 읽거나 복사하지 않습니다. 네트워크 요청과 업로드도 하지 않습니다.

In [ ]:
from pathlib import Path
import importlib.util
import sys

PACKAGE_DIR = Path.cwd().resolve().parent
DATA_DIR = PACKAGE_DIR / '01_data' / 'organizer_dataset'
if not (PACKAGE_DIR / 'run_submission.py').is_file():
    raise RuntimeError('Run this notebook from P?/04_predict')
sys.path.insert(0, str(PACKAGE_DIR))
spec = importlib.util.spec_from_file_location('final_runner', PACKAGE_DIR / 'run_submission.py')
run_submission = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_submission)
from common import bounded_receipt
print({'package': PACKAGE_DIR.name, 'data_present': DATA_DIR.is_dir()})

## 1. 데이터·모델·결정 자산 hash preflight

In [ ]:
preflight = run_submission.preflight(DATA_DIR, PACKAGE_DIR)
bounded_receipt(preflight)

## 2. 모델 추론으로 제출 CSV 생성

In [ ]:
output_path = PACKAGE_DIR / '05_answer' / 'P3_submission.csv'
receipt = run_submission.materialize(DATA_DIR, PACKAGE_DIR, output_path)
bounded_receipt(receipt)

## 3. 제출 계약 확인

In [ ]:
assert receipt['status'] == 'READY_MODEL_INFERENCE_EXACT_NOT_UPLOADED'
assert receipt['candidate_hash_exact'] and receipt['key_order_exact']
assert receipt['prediction_source'] != 'frozen_candidate_csv_copy'
assert output_path.is_file()
print({'status': receipt['status'], 'rows': receipt['rows'], 'sha256': receipt['sha256']})

정확한 열 순서, dtype, 행 수, 홈페이지 제목과 한 줄 요약은 `../06_submission/FORMAT.md`와 `FORM.json`에 고정되어 있습니다.